In [1]:
from transformers import AutoProcessor, AutoModelForCausalLM
import torch


model_path = "model/florence2"
model = (
    AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    .cuda()
    .eval()
)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)

/home/zaln/miniconda3/envs/magiv3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zaln/miniconda3/envs/magiv3/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Florence2LanguageForConditionalGeneration has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warn

In [2]:
import heapq


class DSU:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a, b):
        pa, pb = self.find(a), self.find(b)
        if pa != pb:
            self.p[pb] = pa


def cluster_panels(panels):
    n = len(panels)
    mids = [((y1 + y2) / 2, i) for i, (_, y1, _, y2) in enumerate(panels)]

    panels_sorted = sorted(
        [(y1, y2, i) for i, (_, y1, _, y2) in enumerate(panels)], key=lambda x: x[0]
    )
    mids.sort()

    dsu = DSU(n)
    heap = []  # (r, index)
    j = 0

    for mid, i in mids:
        # 把所有 l <= mid 的区间加入候选
        while j < n and panels_sorted[j][0] <= mid:
            _, r, idx = panels_sorted[j]
            heapq.heappush(heap, (r, idx))
            j += 1

        # 移除 r < mid 的区间
        while heap and heap[0][0] < mid:
            heapq.heappop(heap)

        # 剩下的 heap 里，全是覆盖 mid 的区间
        for _, idx in heap:
            dsu.union(i, idx)

    # 汇总连通分量
    res = {}
    for i in range(n):
        root = dsu.find(i)
        res.setdefault(root, []).append(panels[i])

    return list(res.values())


def get_envelope_boxes(groups):
    res = []
    for group in groups:
        x1s, y1s, x2s, y2s = zip(*group)
        res.append((min(x1s), min(y1s), max(x2s), max(y2s)))

    # 按左上角y坐标排序（从上到下）
    res.sort(key=lambda box: box[1])

    return res

In [3]:
import os
from pathlib import Path
from PIL import Image, ImageDraw

# 确保output_dir存在
input_dir = Path("/home/zaln/下载/1~4")
output_dir = Path("/home/zaln/下载/cropped_images")
output_dir.mkdir(exist_ok=True)

# 所有支持的格式
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp"}

# 收集所有图片
image_paths = []
images = []

for img_path in input_dir.iterdir():
    if img_path.suffix.lower() in image_extensions:
        image = Image.open(img_path).convert("RGB")
        image_paths.append(img_path)
        images.append(image)

if not images:
    print("未找到可处理的图片")
    exit()

print(f"共找到 {len(images)} 张图片")


共找到 707 张图片


In [ ]:
## 测试绘制

# 批量预测
# all_results = model.predict_panels_only(images, processor)


# 处理每张
for img_path, image in zip(image_paths, images):

    panels = model.predict_panels_only([image], processor)[0]["panels"]
    envelope_boxes = get_envelope_boxes(cluster_panels(panels))

    img_draw = image.copy()
    draw = ImageDraw.Draw(img_draw)

    # 绘制检测框
    # for x1, y1, x2, y2 in panels:
    #     draw.rectangle([x1, y1, x2, y2], outline="green", width=3)

    # 绘制裁剪框
    for x1, y1, x2, y2 in envelope_boxes:
        draw.rectangle([x1, y1, x2, y2], outline="green", width=3)

    # 保存
    output_path = output_dir / f"{img_path.stem}{img_path.suffix}"
    img_draw.save(output_path)


In [4]:
## 裁剪
output_dir = Path("/home/zaln/下载/cropped_images")
output_dir.mkdir(exist_ok=True)

for img_path, image in zip(image_paths, images):

    panels = model.predict_panels_only([image], processor)[0]["panels"]
    envelope_boxes = get_envelope_boxes(cluster_panels(panels))

    # 按裁剪框顺序切割并保存图片
    for i, (x1, y1, x2, y2) in enumerate(envelope_boxes):
        # 确保坐标是整数
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        
        # 切割图片
        cropped_image = image.crop((x1, y1, x2, y2))
        
        # 生成输出文件名：原文件名_c1, 原文件名_c2, ...
        output_filename = f"{img_path.stem}_c{i+1}{img_path.suffix}"
        output_path = output_dir / output_filename
        
        # 保存切割后的图片
        cropped_image.save(output_path)
        print(f"已保存: {output_filename}")

/home/zaln/miniconda3/envs/magiv3/lib/python3.10/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


已保存: image00212_c1.jpeg
已保存: image00212_c2.jpeg
已保存: image00439_c1.jpeg
已保存: image00537_c1.jpeg
已保存: image00537_c2.jpeg
已保存: image00385_c1.jpeg
已保存: image00385_c2.jpeg
已保存: image00401_c1.jpeg
已保存: image00401_c2.jpeg
已保存: image00401_c3.jpeg
已保存: image00678_c1.jpeg
已保存: image00678_c2.jpeg
已保存: image00158_c1.jpeg
已保存: image00158_c2.jpeg
已保存: image00158_c3.jpeg
已保存: image00158_c4.jpeg
已保存: image00387_c1.jpeg
已保存: image00387_c2.jpeg
已保存: image00228_c1.jpeg
已保存: image00228_c2.jpeg
已保存: image00133_c1.jpeg
已保存: image00133_c2.jpeg
已保存: image00133_c3.jpeg
已保存: image00510_c1.jpeg
已保存: image00510_c2.jpeg
已保存: image00510_c3.jpeg
已保存: image00252_c1.jpeg
已保存: image00252_c2.jpeg
已保存: image00447_c1.jpeg
已保存: image00313_c1.jpeg
已保存: image00313_c2.jpeg
已保存: image00313_c3.jpeg
已保存: image00610_c1.jpeg
已保存: image00610_c2.jpeg
已保存: image00585_c1.jpeg
已保存: image00585_c2.jpeg
已保存: image00585_c3.jpeg
已保存: image00585_c4.jpeg
已保存: image00283_c1.jpeg
已保存: image00283_c2.jpeg
已保存: image00293_c1.jpeg
已保存: image00293_

crop速度：707张 10m38s

In [ ]:
## 针对双页漫画，拆页
def split_dual_page_comic(image):
    width, height = image.size
    left = image.crop((0, 0, width // 2, height))
    right = image.crop((width // 2, 0, width, height))
    return left, right


import os
from pathlib import Path


def process_and_save_split_images(images, image_paths, output_dir):
    """
    Args:
        images: PIL Image对象列表
        image_paths: 原始图片路径列表
        output_dir: 输出目录
    """
    # 确保输出目录存在
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for idx, (img, img_path) in enumerate(zip(images, image_paths)):
        print(f"处理图片 {idx+1}/{len(images)}: {img_path.name}")

        # 调用分割函数，返回两张裁剪后的图片
        left_img, right_img = split_dual_page_comic(img)

        # 生成输出文件名
        original_stem = img_path.stem  # 不带扩展名的文件名
        original_ext = img_path.suffix  # 文件扩展名

        right_filename = f"{original_stem}_1{original_ext}"
        right_path = output_dir / right_filename

        left_filename = f"{original_stem}_2{original_ext}"
        left_path = output_dir / left_filename

        # 保存图片
        left_img.save(left_path)
        right_img.save(right_path)


output_dir = "/home/zaln/下载/split_images"

process_and_save_split_images(
    images=images, image_paths=image_paths, output_dir=output_dir
)